In [0]:
from pyspark.sql import functions as F 
from pyspark.sql.window import Window

In [0]:
SOURCE_CATALOG = 'covid_19'
SOURCE_SCHEMA = 'gold'

In [0]:
df_bridge_country_vaccine = spark.table(f'{SOURCE_CATALOG}.{SOURCE_SCHEMA}.bridge_country_vaccine').alias('bcv')
df_dim_country = spark.table(f'{SOURCE_CATALOG}.{SOURCE_SCHEMA}.dim_country').alias('dc')
df_dim_vaccine = spark.table(f'{SOURCE_CATALOG}.{SOURCE_SCHEMA}.dim_vaccine').alias('dv')
df_dim_date = spark.table(f'{SOURCE_CATALOG}.{SOURCE_SCHEMA}.dim_date').alias('dd')
df_fact_vaccinations = spark.table(f'{SOURCE_CATALOG}.{SOURCE_SCHEMA}.fact_vaccinations').alias('fv')

## Question 1 - What country/countries use more kinds of vaccines?

In [0]:
df_country_vaccine_count = (
    df_bridge_country_vaccine
    .join(
        df_dim_country,
        on='country_key',
        how='inner'
    )
    .join(
        df_dim_vaccine,
        on='vaccine_key',
        how='inner'
    )
    .groupBy(
        'dc.country_key',
        'dc.country',
        'dc.iso_code'
    )
    .agg(
        F.countDistinct('dv.vaccine_key').alias('vaccine_count'),
        F.array_sort(F.collect_set('dv.vaccine_name')).alias('vaccines')
    )
)

In [0]:
max_vaccine_count = (
    df_country_vaccine_count
    .agg(F.max("vaccine_count").alias("max_vaccine_count"))
    .collect()[0]["max_vaccine_count"]
)

df_answer_1 = (
    df_country_vaccine_count
    .filter(F.col("vaccine_count") == max_vaccine_count)
    .orderBy("country")
    .drop("country_key")
)

display(df_answer_1)

## Question 2 - Top 10 countries that had more vaccinations per month and year


In [0]:
df_monthly_vaccinations = (
    df_fact_vaccinations
    .join(
        df_dim_country,
        on='country_key',
        how='inner'
    )
    .join(
        df_dim_date,
        on='date_key',
        how='inner'
    )
    .groupBy(
        'dd.year',
        'dd.month',
        'dc.country',
        'dc.iso_code'
    )
    .agg(
        F.sum(F.coalesce('fv.daily_vaccinations', F.lit(0))).alias('monthly_vaccinations')
    )
)

In [0]:
window_month_year = Window.partitionBy('year', 'month').orderBy(
    F.desc('monthly_vaccinations')
)

df_answer_2 = (
    df_monthly_vaccinations
    .withColumn('rank', F.row_number().over(window_month_year))
    .filter(F.col('rank') <= 10)
    .select(
        'year',
        'month',
        'rank',
        'country',
        'iso_code',
        'monthly_vaccinations'
    )
    .orderBy('year', 'month', 'rank')
)

display(df_answer_2)

## Question 3 - Top 10 countries by year including all vaccines used

In [0]:
df_yearly_vaccinations = (
    df_fact_vaccinations
    .join(
        df_dim_country,
        on="country_key",
        how="inner"
    )
    .join(
        df_dim_date,
        on="date_key",
        how="inner"
    )
    .groupBy(
        "dd.year",
        "dc.country_key",
        "dc.country",
        "dc.iso_code"
    )
    .agg(
        F.sum(F.coalesce("fv.daily_vaccinations", F.lit(0))).alias("yearly_vaccinations")
    )
).alias('yv')

In [0]:
df_country_vaccine_summary = (
    df_bridge_country_vaccine
    .join(
        df_dim_vaccine,
        on="vaccine_key",
        how="inner"
    )
    .groupBy("bcv.country_key")
    .agg(
        F.countDistinct("dv.vaccine_key").alias("vaccine_count"),
        F.array_sort(F.collect_set("dv.vaccine_name")).alias("vaccines")
    )
).alias('cvs')

In [0]:
df_yearly_with_vaccines = (
    df_yearly_vaccinations
    .join(
        df_country_vaccine_summary,
        on="country_key",
        how="left"
    )
    .fillna({"vaccine_count": 0})
)

In [0]:
window_year = Window.partitionBy("year").orderBy(
    F.desc("vaccine_count"),
    F.desc("yearly_vaccinations")
)

df_answer_3 = (
    df_yearly_with_vaccines
    .withColumn("rank", F.row_number().over(window_year))
    .filter(F.col("rank") <= 10)
    .select(
        "year",
        "rank",
        "country",
        "iso_code",
        "vaccine_count",
        "yearly_vaccinations",
        "vaccines"
    )
    .orderBy("year", "rank")
)

display(df_answer_3)